# Gating a change on experiment scores (CI/CD) 🚦

The previous notebooks ran experiments interactively. This one wires experiments into a **continuous
integration** workflow: every commit runs the same experiment against the same dataset, and a change
is only allowed through if its scores are **at least as good as the baseline**.

The flow we'll build:

1. **Run** an experiment on the current commit.
2. **Pin it as the baseline** — the run every later commit is measured against.
3. **Change the prompt and commit it**, exactly as you would in a pull request.
4. **Re-run**, fetch the baseline back from Datadog, **compare, and gate** on the scores improving.

Two SDK features make this work. Every experiment is **automatically tagged with the `git.commit.sha`
it ran on** — no bookkeeping on your side. And
[`LLMObs.list_experiments()`](https://github.com/DataDog/dd-trace-py/pull/19329) lets a later job look
up "the run for commit `abc123`" and read its `aggregate_data` — average eval scores, error counts,
token costs — without keeping any state of its own between CI jobs. Datadog *is* the state.

## 0. Set up

`list_experiments()` was added in [dd-trace-py#19329](https://github.com/DataDog/dd-trace-py/pull/19329).
If the check below fails, upgrade `ddtrace` to a version that includes it.

This notebook also runs `git commit` in step 5, so run it from inside a git checkout, on a branch you
don't mind adding a commit to.

In [ ]:
import json
import os
import subprocess
import time

from dotenv import load_dotenv
# Load environment variables from the .env file.
load_dotenv(override=True)

from typing import Any, Dict, Optional

from ddtrace.llmobs import LLMObs

from openai import OpenAI

LLMObs.enable(api_key=os.getenv("DD_API_KEY"), app_key=os.getenv("DD_APPLICATION_KEY"),  project_name="Onboarding", ml_app="Onboarding-ML-App")

oai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

if not hasattr(LLMObs, "list_experiments"):
    raise RuntimeError(
        "This notebook needs LLMObs.list_experiments(), added in dd-trace-py#19329. "
        "Please upgrade ddtrace."
    )

### Git helpers

In real CI these come for free — GitHub Actions gives you `GITHUB_SHA`, GitLab gives you
`CI_COMMIT_SHA`. Here we read them from the local checkout so the notebook is self-contained.

We don't have to tag the experiment with the SHA ourselves — **the SDK does it**. Every experiment is
automatically tagged with `git.commit.sha` and `git.repository_url`, resolved in `Experiment.__init__`
from `DD_GIT_COMMIT_SHA` / `DD_GIT_REPOSITORY_URL` if set, falling back to `git` in the working
directory. That resolution happens *per experiment*, not once at `LLMObs.enable()`, so the commit we
create halfway through this notebook is picked up on its own.

We read the SHA locally only so we know **what to look up** afterwards. In CI you'd take it from
`$GITHUB_SHA` instead — and setting `DD_GIT_COMMIT_SHA` is the more robust way to do that, since a CI
checkout is often shallow or detached.

Worth remembering that a SHA only identifies the *commit*: run an experiment against a dirty working
tree and the tag will claim a state that isn't what actually ran. CI gets this right for free by
building from a clean checkout.

In [ ]:
def git_sha() -> str:
    """The commit the working tree is on. In CI you'd read $GITHUB_SHA instead."""
    return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()


print("commit:", git_sha())

## 1. The dataset

CI needs the *same* dataset on every run, so we pull it from Datadog by name rather than rebuilding
it. We'll reuse the capitals dataset — created from `capitals.csv` the first time this runs, pulled
on every run after that.

In [ ]:
DATASET_NAME = "capitals-of-the-world"

try:
    dataset = LLMObs.pull_dataset(dataset_name=DATASET_NAME)
except Exception:
    # First run in this org: seed the dataset from the CSV.
    dataset = LLMObs.create_dataset_from_csv(
        csv_path="./capitals.csv",
        dataset_name=DATASET_NAME,
        input_data_columns=["question"],
        expected_output_columns=["answer"],
        metadata_columns=["category", "difficulty"],
        description="a list of inputs and outputs describing capitals of the world",
    )

dataset.as_dataframe()

## 2. The task under test

For a commit to be able to change the behaviour under test, the prompt has to live in a **file that
git tracks** — not in a notebook cell. Ours is in `prompts/capital_prompt.txt`, and the task reads it
from disk on every run.

This is the whole point of the setup: the thing CI measures is whatever is checked in at that commit.

In [ ]:
PROMPT_PATH = os.path.abspath(os.path.join("prompts", "capital_prompt.txt"))


def load_prompt() -> str:
    with open(PROMPT_PATH) as f:
        return f.read().strip()


def generate_capital(input_data: Dict[str, Any], config: Dict[str, Any]) -> str:
    output = oai_client.chat.completions.create(
        model=config["model"],
        messages=[
            {"role": "system", "content": load_prompt()},
            {"role": "user", "content": input_data["question"]},
        ],
        temperature=config["temperature"],
    )
    return output.choices[0].message.content


def exact_match(input_data, output_data, expected_output):
    return expected_output == output_data


def contains_answer(input_data, output_data, expected_output):
    return expected_output in output_data


print(load_prompt())

### One helper to run the experiment

Both CI runs are the *same* experiment — same name, same dataset, same evaluators — differing only in
the commit they ran on. `LLMObs.experiment()` generates a unique run name per invocation, so the
shared `EXPERIMENT_NAME` is what ties the history together and what we filter on later.

In [ ]:
EXPERIMENT_NAME = "capitals-ci"
CONFIG = {"model": "gpt-4.1-nano", "temperature": 0}


def run_experiment(ci_stage: str):
    """Run the experiment — exactly what CI does on every push. The git.commit.sha tag is added by the SDK."""
    experiment = LLMObs.experiment(
        name=EXPERIMENT_NAME,
        dataset=dataset,
        task=generate_capital,
        evaluators=[exact_match, contains_answer],
        config=CONFIG,
        description=f"capitals accuracy gate ({ci_stage})",
        tags={"ci.stage": ci_stage},  # only our own tag; git tags are automatic
    )
    experiment.run(jobs=5)
    return experiment

## 3. Run the baseline

This stands in for the job that runs on your default branch. Note the commit SHA it prints — that's
the handle we'll use to fetch this run back later.

In [ ]:
BASELINE_SHA = git_sha()

baseline_experiment = run_experiment(ci_stage="baseline")

print("baseline commit:", BASELINE_SHA)
baseline_experiment.url

## 4. Pin it as the baseline

There's no "baseline" flag on an experiment — the baseline is just **a commit SHA you decided to
compare against**, which in practice is the SHA your branch was cut from (`git merge-base`) or
whatever is currently on `main`.

`list_experiments()` turns that SHA back into a run. Two details matter for CI:

- Filter on `metadata_filter={"tags": ["git.commit.sha:<sha>"]}`. Experiments created by the SDK store
  only `tags` under metadata, so tag containment is how commit lookups work.
- Only compare runs whose `status` is `completed`. A `running`, `failed` or `interrupted` run has
  missing or partial `aggregate_data`, and gating on it would compare against noise. `list_experiments()`
  has no status filter, so we filter client-side — which is also why `max_results` is worth bounding:
  the cap applies to rows fetched, before we discard the incomplete ones.

`aggregate_data` is computed server-side shortly after a run finishes, so the helper below polls
until it shows up rather than assuming it's there the instant `run()` returns.

In [ ]:
def find_run(commit_sha: str, timeout_s: int = 180, poll_s: int = 10) -> Dict[str, Any]:
    """Return the most recent completed run of EXPERIMENT_NAME for a commit, waiting for aggregates."""
    deadline = time.time() + timeout_s
    while True:
        runs = LLMObs.list_experiments(
            experiment_name=EXPERIMENT_NAME,
            metadata_filter={"tags": [f"git.commit.sha:{commit_sha}"]},
            max_results=20,  # results are newest-first; don't walk the whole project history
        )
        # Skip anything still running or failed — its aggregate_data is partial or absent.
        completed = [r for r in runs if r.get("status") == "completed" and r.get("aggregate_data")]
        if completed:
            return completed[0]
        if time.time() >= deadline:
            statuses = [r.get("status") for r in runs] or ["<no runs found>"]
            raise TimeoutError(
                f"No completed run with aggregates for {commit_sha[:8]} after {timeout_s}s (saw: {statuses})"
            )
        time.sleep(poll_s)


baseline_run = find_run(BASELINE_SHA)

print("run id:", baseline_run["id"], "| status:", baseline_run["status"])

# Proof the SDK tagged the run for us — we never set these.
print("git tags:", [t for t in baseline_run.get("tags") or [] if t.startswith("git.")])

print(json.dumps(baseline_run["aggregate_data"], indent=2))

### Reading scores out of `aggregate_data`

`aggregate_data` carries per-evaluator distributions under `evaluations`, alongside run-level totals
(`total_errors`, `total_tokens`, `estimated_total_cost`, ...). The helper below reduces each evaluator
to a single number: an average where the backend provides one, otherwise a pass rate over a
boolean/categorical distribution.

Compare it against the raw payload printed above and adjust if your evaluators report a shape it
doesn't recognise.

In [ ]:
def _to_score(value: Any) -> Optional[float]:
    """Reduce one evaluator's aggregate entry to a single comparable number."""
    if isinstance(value, bool):
        return float(value)
    if isinstance(value, (int, float)):
        return float(value)
    if not isinstance(value, dict):
        return None
    for key in ("avg", "mean", "average", "value", "score"):
        if isinstance(value.get(key), (int, float)) and not isinstance(value.get(key), bool):
            return float(value[key])
    # Boolean/categorical distribution, e.g. {"true": 3, "false": 1} -> pass rate.
    counts = {str(k).lower(): n for k, n in value.items() if isinstance(n, (int, float))}
    total = sum(counts.values())
    if total:
        return (counts.get("true", 0) + counts.get("pass", 0)) / total
    return None


def eval_scores(run: Dict[str, Any]) -> Dict[str, float]:
    """Map evaluator name -> score for a run returned by list_experiments()."""
    aggregate = run.get("aggregate_data") or {}
    evaluations = aggregate.get("evaluations") or {}

    buckets: Dict[str, Any] = {}
    for group in ("custom", "summary"):
        buckets.update(evaluations.get(group) or {})
    if not buckets:
        # Flatter shape: evaluator names at the top level of aggregate_data.
        buckets = {
            k: v for k, v in aggregate.items()
            if isinstance(v, dict) and k != "evaluations"
        }

    scores = {name: _to_score(v) for name, v in buckets.items()}
    return {name: score for name, score in scores.items() if score is not None}


baseline_scores = eval_scores(baseline_run)
baseline_scores

## 5. Change the prompt and commit

This is the pull request. We rewrite the tracked prompt file and commit it, which gives us a **new
commit SHA** — the one the candidate run will be tagged with.

The current prompt lets the model answer in a full sentence, so `exact_match` fails on every row. The
new one constrains it to just the city name.

In [ ]:
NEW_PROMPT = (
    "You will respond only with the name of the capital city, nothing else. "
    "No sentence, no punctuation, no explanation."
)

with open(PROMPT_PATH, "w") as f:
    f.write(NEW_PROMPT + "\n")

subprocess.run(["git", "add", PROMPT_PATH], check=True)
subprocess.run(["git", "commit", "-m", "Constrain capitals prompt to a bare city name"], check=True)

CANDIDATE_SHA = git_sha()
print("baseline commit: ", BASELINE_SHA)
print("candidate commit:", CANDIDATE_SHA)
assert CANDIDATE_SHA != BASELINE_SHA, "expected the commit to create a new SHA"

## 6. Re-run on the new commit

Identical code path — same experiment name, same dataset, same evaluators, same call. The only thing
that changed is the prompt on disk and the commit we're on, and the SDK notices the latter by itself.

In [ ]:
print("prompt now:", load_prompt())

candidate_experiment = run_experiment(ci_stage="candidate")

# The SDK re-resolved git metadata for this experiment and picked up the new commit on its own:
# find_run() locates it by the SHA we never passed in.
candidate_run = find_run(CANDIDATE_SHA)
print("found run", candidate_run["id"], "for", CANDIDATE_SHA[:8])
candidate_experiment.url

## 7. Compare and gate

The gate itself. It fetches both runs back from Datadog by commit SHA — note that it never reuses the
in-memory `baseline_experiment` object, because in real CI the baseline was produced by a *different
job on a different machine*, possibly days earlier.

The policy encoded here: **no evaluator may regress**, at least one must improve, and the run must
not produce more errors than the baseline. Tighten or loosen `MIN_DELTA` to taste — on a small
dataset a tiny score movement is noise, so a real gate usually wants either a margin or more rows.

In [ ]:
MIN_DELTA = 0.0  # a regression larger than this fails the build


def gate(baseline_sha: str, candidate_sha: str) -> bool:
    baseline, candidate = find_run(baseline_sha), find_run(candidate_sha)
    before, after = eval_scores(baseline), eval_scores(candidate)

    shared = sorted(set(before) & set(after))
    if not shared:
        raise ValueError(
            f"No evaluators in common: baseline={sorted(before)} candidate={sorted(after)}. "
            "Check the raw aggregate_data printed above and adjust eval_scores() to its shape."
        )

    regressions, improvements = [], []
    print(f"{'evaluator':<24} {'baseline':>10} {'candidate':>10} {'delta':>8}")
    for name in shared:
        delta = after[name] - before[name]
        print(f"{name:<24} {before[name]:>10.3f} {after[name]:>10.3f} {delta:>+8.3f}")
        if delta < -MIN_DELTA:
            regressions.append(f"{name} {before[name]:.3f} -> {after[name]:.3f}")
        elif delta > 0:
            improvements.append(name)

    errors_before = (baseline.get("aggregate_data") or {}).get("total_errors") or 0
    errors_after = (candidate.get("aggregate_data") or {}).get("total_errors") or 0
    print(f"\n{'total_errors':<24} {errors_before:>10} {errors_after:>10}")
    if errors_after > errors_before:
        regressions.append(f"total_errors {errors_before} -> {errors_after}")

    print()
    if regressions:
        print("FAIL — regressed: " + "; ".join(regressions))
        return False
    if not improvements:
        print("FAIL — no evaluator improved; nothing to justify the change")
        return False
    print("PASS — improved: " + ", ".join(improvements))
    return True


passed = gate(BASELINE_SHA, CANDIDATE_SHA)

# In a CI job this is the exit code. A non-zero exit is what blocks the merge.
if not passed:
    raise SystemExit(1)

## 8. Wiring it into CI

Everything above collapses into one script: pull the dataset, run the experiment tagged with
`$GITHUB_SHA`, fetch the baseline for the merge-base, compare, exit non-zero on a regression. No
artifacts to pass between jobs — the baseline is fetched from Datadog by SHA.

```yaml
name: llm-eval-gate

on: pull_request

jobs:
  gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0  # needed for git merge-base

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - run: pip install -r experiments/notebooks/requirements.in

      - name: Run experiment and gate on scores
        env:
          DD_API_KEY: ${{ secrets.DD_API_KEY }}
          DD_APPLICATION_KEY: ${{ secrets.DD_APPLICATION_KEY }}
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          # ddtrace reads this to tag the experiment; more reliable than a shallow CI checkout
          DD_GIT_COMMIT_SHA: ${{ github.event.pull_request.head.sha }}
          DD_GIT_REPOSITORY_URL: ${{ github.server_url }}/${{ github.repository }}
          BASELINE_SHA: ${{ github.event.pull_request.base.sha }}
        run: python experiments/scripts/eval_gate.py
```

A few things worth deciding before you turn a gate like this on:

- **Baseline choice.** `github.event.pull_request.base.sha` compares against the commit the branch was
  cut from. If your default branch also runs the experiment on every merge, you can instead take the
  newest completed run on `main` regardless of SHA.
- **Dataset size.** Three rows is enough to demonstrate the mechanics and far too few to gate on — one
  flaky row is a 33% swing. Grow the dataset before the gate means anything.
- **Non-determinism.** Even at `temperature=0` scores move between runs. Either require a margin
  (`MIN_DELTA`) or use `runs=N` on the experiment and compare averages.
- **Cost.** Every gated PR is a full dataset pass through your model. `estimated_total_cost` in
  `aggregate_data` tells you what each run costs.

## Done 🎉

We ran an experiment, pinned it as a baseline by commit SHA, changed a checked-in prompt, re-ran, and
blocked-or-passed the change on the score delta — with `LLMObs.list_experiments()` as the only thing
carrying state between the two runs.

The prompt file is now modified and committed in your checkout. To rerun this notebook from a clean
slate, undo that commit:

```bash
git revert --no-edit HEAD    # or: git reset --hard HEAD~1
```